# Nautiq — ML clasificación JIT / NO_JIT

**Unidad:** 1 fila = 1 `port_call` válido de la Gold.

**Target**
- `JIT = 1`: `actual_wait_hours <= median_wait_hours` de su `destination_port_code + vessel_length_band`.
- `NO_JIT = 0`: `actual_wait_hours > median_wait_hours`.

La mediana solo construye la etiqueta; **no entra como feature**.

**Modelos**
1. Logistic Regression — baseline lineal e interpretable.
2. Random Forest — bagging y relaciones no lineales.
3. XGBoost — gradient boosting para datos tabulares.

**Validación**
- `StratifiedGroupKFold` de **10 folds** por `mmsi`.
- Cada modelo se entrena 10 veces y cada fold actúa una vez como test.
- Es una única validacin cruzada de 10 folds.

**Reducción final**
Tras elegir el mejor modelo, se estima la importancia de las variables dentro del train de cada fold. Se reentrena una versión reducida y se evalúa en el test externo del fold para comprobar si simplificar mantiene o mejora el resultado.


In [0]:
%run ../setup_nautiq_dev


# Nautiq - setup del entorno

Configuración centralizada para el flujo activo del TFM:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado técnico de Auto Loader.
- Tablas Silver DEV.
- Gold histórico de port calls.
- Gold de baseline histórico de espera.
- Gold de predicciones JIT actuales.
- Modelo ML registrado en Unity Catalog con alias `Champion`.
- Cambio futuro entre tablas administradas y ADLS externo.

### Notebooks activos

- `silver_ais_positions_dev`
- `silver_ais_static_dev`
- `gold_vessel_port_calls_jit`
- `gold_waiting_avg_per_length`
- `ml_vessel_jit_classification`
- `gold_vessel_jit_current_predictions`


DataFrame[]

NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 17 elementos encontrados
[OK] ais_static: 25 elementos encontrados

Silver DEV tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold tables:
  - masterxyz002dbr.gold.vessel_port_calls_analytics
  - masterxyz002dbr.gold.waiting_avg_per_length
  - masterxyz002dbr.gold.vessel_jit_current_predictions

Registered ML model:
  - masterxyz002dbr.gold.vessel_jit_classifier@Champion

Active notebooks:
  - silver_ais_positions_dev
  - silver_ais_static_dev
  - gold_vessel_port_calls_jit
  - gold_waiting_avg_per_length
  - ml_vessel_jit_classification
  - gold_vessel_jit_current_predictions

[OK] Setup completado correctamente.


In [0]:
from pyspark.sql import functions as F

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, make_scorer
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from xgboost import XGBClassifier

GOLD_TABLE = vessel_port_calls_target_table
BASELINE_TABLE = waiting_avg_per_length_target_table

TARGET = "jit_label"
GROUP_COLUMN = "mmsi"
ID_COLUMN = "port_call_id"

N_SPLITS = 10
BASE_SEED = 42
MAX_NULL_PCT = 0.50
MAX_NUMERIC_CORRELATION = 0.90

# Segunda fase: reducción del modelo ganador.
N_PERMUTATION_REPEATS = 5
MIN_PERMUTATION_IMPORTANCE = 0.01
MIN_SELECTED_FEATURES = 3
MIN_FEATURE_FOLDS = 6
INNER_VALIDATION_SIZE = 0.25

print("Gold:", GOLD_TABLE)
print("Baseline:", BASELINE_TABLE)


Gold: masterxyz002dbr.gold.vessel_port_calls_analytics
Baseline: masterxyz002dbr.gold.waiting_avg_per_length


## 1. Dataset ML y etiqueta JIT

Se usan únicamente escalas con espera real observada. La tabla `waiting_avg_per_length` aporta la mediana histórica por puerto y rango de eslora.


In [0]:
if not spark.catalog.tableExists(GOLD_TABLE): raise RuntimeError(f"No existe {GOLD_TABLE}")
if not spark.catalog.tableExists(BASELINE_TABLE): raise RuntimeError(f"No existe {BASELINE_TABLE}")

gold = spark.table(GOLD_TABLE)
baseline = spark.table(BASELINE_TABLE).select("destination_port_code", "vessel_length_band", "median_wait_hours")

ml_spark = gold.filter(F.col("actual_wait_hours").isNotNull()).join(F.broadcast(baseline), ["destination_port_code", "vessel_length_band"], "inner").filter(F.col("median_wait_hours").isNotNull())
ml_spark = ml_spark.withColumn(TARGET, F.when(F.col("actual_wait_hours") <= F.col("median_wait_hours"), F.lit(1)).otherwise(F.lit(0)))
ml_spark = ml_spark.withColumn("jit_class", F.when(F.col(TARGET) == 1, F.lit("JIT")).otherwise(F.lit("NO_JIT")))

display(ml_spark.agg(F.count("*").alias("port_calls"), F.countDistinct("mmsi").alias("vessels")))
display(ml_spark.groupBy("jit_class", TARGET).agg(F.count("*").alias("port_calls"), F.countDistinct("mmsi").alias("vessels")).orderBy(TARGET))
display(ml_spark.groupBy("destination_port_code", "vessel_length_band", "jit_class").count().orderBy("destination_port_code", "vessel_length_band", "jit_class"))


port_calls,vessels
169,110


jit_class,jit_label,port_calls,vessels
NO_JIT,0,84,61
JIT,1,85,70


destination_port_code,vessel_length_band,jit_class,count
ESALG,100-200,JIT,4
ESALG,100-200,NO_JIT,4
ESALG,200-300,JIT,13
ESALG,200-300,NO_JIT,12
ESALG,300-600,JIT,9
ESALG,300-600,NO_JIT,10
ESBCN,100-200,JIT,10
ESBCN,100-200,NO_JIT,9
ESBCN,200-300,JIT,11
ESBCN,200-300,NO_JIT,12


## 2. Variables candidatas

Se utilizan únicamente variables disponibles antes del fondeo/servicio.

Se descartan identificadores, variables posteriores a la operación y variables que forman parte del cálculo del target.

El control posterior de correlación se mantiene para detectar nuevas redundancias cuando aumente el histórico.


In [0]:
numeric_candidates = [
    "vessel_length_meters",
    "vessel_beam_meters",
    "vessel_draught_meters",
    "speed_over_ground_knots",
    "course_over_ground_degrees",
    "true_heading_degrees",
    "distance_to_service_nm",
]

categorical_candidates = [
    "destination_port_code",
    "ship_type_category",
    "navigation_status_code",
]

numeric_candidates = [c for c in numeric_candidates if c in ml_spark.columns]
categorical_candidates = [c for c in categorical_candidates if c in ml_spark.columns]
candidate_features = numeric_candidates + categorical_candidates

print("Numéricas:", numeric_candidates)
print("Categóricas:", categorical_candidates)


Numéricas: ['vessel_length_meters', 'vessel_beam_meters', 'vessel_draught_meters', 'speed_over_ground_knots', 'course_over_ground_degrees', 'true_heading_degrees', 'distance_to_service_nm']
Categóricas: ['destination_port_code', 'ship_type_category', 'navigation_status_code']


## 3. Pasar solo las escalas ML a Pandas

Con unas pocas centenas de `port_call`, trabajar en Pandas/scikit-learn es más rápido y barato que ejecutar repetidamente pipelines Spark ML.


In [0]:
selected_columns = [ID_COLUMN, GROUP_COLUMN, TARGET, "jit_class"] + candidate_features
pdf = ml_spark.select(*selected_columns).dropDuplicates([ID_COLUMN]).toPandas()

pdf["course_over_ground_degrees"] = pd.to_numeric(pdf["course_over_ground_degrees"], errors="coerce").replace(360.0, np.nan) if "course_over_ground_degrees" in pdf else np.nan
pdf["true_heading_degrees"] = pd.to_numeric(pdf["true_heading_degrees"], errors="coerce").replace(511.0, np.nan) if "true_heading_degrees" in pdf else np.nan

for c in numeric_candidates:
    if c in pdf.columns: pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

pdf = pdf.dropna(subset=[TARGET, GROUP_COLUMN]).copy()
pdf[TARGET] = pdf[TARGET].astype(int)

print("Port calls:", len(pdf))
print("Buques:", pdf[GROUP_COLUMN].nunique())
print("Clases:", pdf["jit_class"].value_counts().to_dict())


Port calls: 169
Buques: 110
Clases: {'JIT': 85, 'NO_JIT': 84}


## 4. Control rápido de calidad y redundancia

Antes de entrenar solo se descartan:
- variables con más del 50 % de nulos;
- variables constantes;
- una variable de cada par numérico con `|Spearman| >= 0.90`.

Pearson, Spearman y Mutual Information se usan como **diagnóstico**, no como corte definitivo. Una variable con poca relación individual puede ser útil combinada con otras en Random Forest o XGBoost.

La reducción más fuerte se realiza después de elegir el modelo ganador y se valida de nuevo.


In [0]:
quality_rows = []
for c in candidate_features:
    null_pct = float(pdf[c].isna().mean())
    nunique = int(pdf[c].nunique(dropna=True))
    reason = f">{MAX_NULL_PCT:.0%} nulos" if null_pct > MAX_NULL_PCT else ("Constante" if nunique <= 1 else None)
    quality_rows.append({"variable": c, "tipo": "numeric" if c in numeric_candidates else "categorical", "nulos_pct": round(null_pct * 100, 2), "valores_unicos": nunique, "descartar": reason is not None, "motivo": reason})

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

quality_valid = quality_df.loc[~quality_df["descartar"], "variable"].tolist()
numeric_valid = [c for c in numeric_candidates if c in quality_valid]
categorical_valid = [c for c in categorical_candidates if c in quality_valid]


variable,tipo,nulos_pct,valores_unicos,descartar,motivo
vessel_length_meters,numeric,0.0,126,false,null
vessel_beam_meters,numeric,0.0,44,false,null
vessel_draught_meters,numeric,0.0,80,false,null
speed_over_ground_knots,numeric,4.14,61,false,null
course_over_ground_degrees,numeric,12.43,121,false,null
true_heading_degrees,numeric,34.32,9,false,null
distance_to_service_nm,numeric,4.14,64,false,null
destination_port_code,categorical,0.0,3,false,null
ship_type_category,categorical,0.0,1,true,Constante
navigation_status_code,categorical,4.14,3,false,null


### Relación de las variables con el target
- **Pearson / point-biserial**: mide la relación lineal entre una variable numérica y el target binario (`0 = NO_JIT`, `1 = JIT`).  
  Valores cercanos a `1` indican asociación positiva con JIT, cercanos a `-1` asociación con NO_JIT y cercanos a `0` poca relación lineal.

- **Spearman**: mide si existe una relación monotónica entre la variable y el target utilizando el orden de los valores. Puede detectar relaciones que no sean estrictamente lineales.

- **abs_spearman**: valor absoluto de Spearman. Se utiliza únicamente para ordenar las variables según la intensidad de su relación, sin importar si esta es positiva o negativa.


In [0]:
target_relation = []
for c in numeric_valid:
    tmp = pdf[[c, TARGET]].dropna()
    pearson = tmp[c].corr(tmp[TARGET], method="pearson") if len(tmp) >= 3 and tmp[c].nunique() > 1 else np.nan
    spearman = tmp[c].corr(tmp[TARGET], method="spearman") if len(tmp) >= 3 and tmp[c].nunique() > 1 else np.nan
    target_relation.append({"variable": c, "pearson_point_biserial": pearson, "spearman": spearman, "abs_spearman": abs(spearman) if pd.notna(spearman) else np.nan})

target_corr_df = pd.DataFrame(target_relation).sort_values("abs_spearman", ascending=False)
display(target_corr_df.round(4))


variable,pearson_point_biserial,spearman,abs_spearman
speed_over_ground_knots,0.4577,0.4548,0.4548
distance_to_service_nm,-0.0472,0.4362,0.4362
vessel_beam_meters,0.1146,0.1148,0.1148
true_heading_degrees,0.0598,0.065,0.065
vessel_draught_meters,-0.0161,-0.0032,0.0032
course_over_ground_degrees,-0.0539,0.0029,0.0029
vessel_length_meters,0.0075,0.0019,0.0019


### Mutual Information

La Mutual Information mide cuánta información aporta cada variable para conocer la clase `JIT / NO_JIT`.

A diferencia de la correlación, puede detectar relaciones no lineales y también permite evaluar variables categóricas.

Antes del cálculo:
- las variables numéricas con valores nulos se completan con su mediana;
- las variables categóricas se convierten temporalmente a códigos numéricos.

Interpretación:
- `MI = 0`: la variable prácticamente no aporta información sobre el target;
- cuanto mayor sea el valor, mayor dependencia existe entre la variable y `jit_label`.

La Mutual Information no tiene un máximo fijo como la correlación, por lo que se utiliza principalmente para comparar la importancia relativa entre las variables del mismo dataset.


In [0]:
mi_data = pd.DataFrame(index=pdf.index)
discrete_mask = []

for c in numeric_valid:
    x = pd.to_numeric(pdf[c], errors="coerce")
    mi_data[c] = x.fillna(x.median())
    discrete_mask.append(False)

for c in categorical_valid:
    mi_data[c] = pd.factorize(pdf[c].astype("object").where(pdf[c].notna(), "__MISSING__"))[0]
    discrete_mask.append(True)

mi_values = mutual_info_classif(mi_data, pdf[TARGET], discrete_features=discrete_mask, random_state=BASE_SEED) if len(mi_data.columns) else np.array([])
mi_df = pd.DataFrame({"variable": mi_data.columns, "mutual_information": mi_values}).sort_values("mutual_information", ascending=False)
display(mi_df.round(4))


variable,mutual_information
navigation_status_code,0.116
distance_to_service_nm,0.0938
speed_over_ground_knots,0.0412
course_over_ground_degrees,0.0334
destination_port_code,2.0E-4
vessel_length_meters,0.0
vessel_beam_meters,0.0
vessel_draught_meters,0.0
true_heading_degrees,0.0


In [0]:
corr_matrix = pdf[numeric_valid].corr(method="spearman") if numeric_valid else pd.DataFrame()
display(corr_matrix.round(3))

mi_lookup = mi_df.set_index("variable")["mutual_information"].to_dict()
high_corr_pairs = []

for i, c1 in enumerate(numeric_valid):
    for c2 in numeric_valid[i + 1:]:
        r = corr_matrix.loc[c1, c2]
        if pd.notna(r) and abs(r) >= MAX_NUMERIC_CORRELATION: high_corr_pairs.append({"variable_1": c1, "variable_2": c2, "spearman": float(r)})

display(pd.DataFrame(high_corr_pairs).sort_values("spearman", key=lambda s: s.abs(), ascending=False) if high_corr_pairs else pd.DataFrame(columns=["variable_1", "variable_2", "spearman"]))


vessel_length_meters,vessel_beam_meters,vessel_draught_meters,speed_over_ground_knots,course_over_ground_degrees,true_heading_degrees,distance_to_service_nm
1.0,0.319,0.277,0.091,0.034,0.13,0.093
0.319,1.0,0.264,0.144,0.066,0.224,0.161
0.277,0.264,1.0,-0.034,0.004,0.086,-0.03
0.091,0.144,-0.034,1.0,0.013,0.046,0.851
0.034,0.066,0.004,0.013,1.0,0.905,-0.001
0.13,0.224,0.086,0.046,0.905,1.0,0.066
0.093,0.161,-0.03,0.851,-0.001,0.066,1.0


variable_1,variable_2,spearman
course_over_ground_degrees,true_heading_degrees,0.9053800364668027


In [0]:
# Selección previa: solo se elimina redundancia fuerte entre numéricas.
# Si dos variables tienen |Spearman| >= 0.90, se conserva la de mayor Mutual Information.

numeric_selected = numeric_valid.copy()
correlation_drops = []

for pair in high_corr_pairs:
    c1, c2 = pair["variable_1"], pair["variable_2"]
    if c1 not in numeric_selected or c2 not in numeric_selected:
        continue

    drop_col = c1 if mi_lookup.get(c1, 0.0) < mi_lookup.get(c2, 0.0) else c2
    keep_col = c2 if drop_col == c1 else c1

    numeric_selected.remove(drop_col)
    correlation_drops.append({
        "descartada": drop_col,
        "conservada": keep_col,
        "spearman": round(pair["spearman"], 4)
    })

categorical_selected = categorical_valid.copy()
final_features = numeric_selected + categorical_selected

if correlation_drops:
    display(pd.DataFrame(correlation_drops))

print("Features iniciales para comparar los 3 modelos:", final_features)
print("Número de features:", len(final_features))

if not final_features:
    raise RuntimeError("No queda ninguna feature válida.")


descartada,conservada,spearman
true_heading_degrees,course_over_ground_degrees,0.9054


Features iniciales para comparar los 3 modelos: ['vessel_length_meters', 'vessel_beam_meters', 'vessel_draught_meters', 'speed_over_ground_knots', 'course_over_ground_degrees', 'distance_to_service_nm', 'destination_port_code', 'navigation_status_code']
Número de features: 8


## 5. Preparación para los tres modelos

Las numéricas se imputan por mediana. Logistic Regression además las estandariza. Las categóricas se imputan por moda y se codifican con One-Hot Encoding.


In [0]:
X = pdf[final_features].copy()
y = pdf[TARGET].astype(int).copy()
groups = pdf[GROUP_COLUMN].astype(str).copy()

def make_preprocessor(features, scale_numeric):
    numeric_features = [c for c in features if c in numeric_selected]
    categorical_features = [c for c in features if c in categorical_selected]

    transformers = []

    if numeric_features:
        num_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            num_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(num_steps), numeric_features))

    if categorical_features:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        transformers.append(("cat", cat_pipe, categorical_features))

    return ColumnTransformer(transformers, remainder="drop")

def build_model(model_name, features):
    if model_name == "Logistic Regression":
        estimator = LogisticRegression(
            solver="liblinear", penalty="l1", C=1.0,
            class_weight="balanced", max_iter=1000, random_state=BASE_SEED
        )
        scale_numeric = True

    elif model_name == "Random Forest":
        estimator = RandomForestClassifier(
            n_estimators=150, max_depth=4, min_samples_leaf=3,
            class_weight="balanced", random_state=BASE_SEED, n_jobs=2
        )
        scale_numeric = False

    elif model_name == "XGBoost":
        estimator = XGBClassifier(
            n_estimators=80, max_depth=2, learning_rate=0.05,
            min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss",
            random_state=BASE_SEED, n_jobs=2
        )
        scale_numeric = False

    else:
        raise ValueError(f"Modelo no reconocido: {model_name}")

    return Pipeline([
        ("preprocessor", make_preprocessor(features, scale_numeric)),
        ("model", estimator)
    ])

def build_models(features):
    return {
        name: build_model(name, features)
        for name in ["Logistic Regression", "Random Forest", "XGBoost"]
    }

print("X shape:", X.shape)
print("Buques:", groups.nunique())
print("Target:", y.value_counts().sort_index().to_dict(), "(0=NO_JIT, 1=JIT)")


X shape: (169, 8)
Buques: 110
Target: {0: 84, 1: 85} (0=NO_JIT, 1=JIT)


## 6. Validación cruzada: 10 folds por MMSI

Se usa `StratifiedGroupKFold`:
- mantiene separados los `mmsi` entre train y test;
- intenta conservar el equilibrio JIT / NO_JIT;
- genera 10 particiones: cada modelo se entrena **10 veces**.

Esto es **10-fold cross-validation**, no validación cruzada repetida.


In [0]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED)
results = []
confusions = {name: np.zeros((2, 2), dtype=int) for name in build_models(final_features)}

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    train_groups, test_groups = set(groups.iloc[train_idx]), set(groups.iloc[test_idx])
    if train_groups.intersection(test_groups): raise RuntimeError(f"Leakage de MMSI en fold {fold}")
    if y_train.nunique() < 2 or y_test.nunique() < 2: raise RuntimeError(f"Fold {fold} no contiene ambas clases.")

    for model_name, model in build_models(final_features).items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        confusions[model_name] += cm

        results.append({
            "fold": fold,
            "model": model_name,
            "train_calls": len(train_idx),
            "test_calls": len(test_idx),
            "train_vessels": len(train_groups),
            "test_vessels": len(test_groups),
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "precision_jit": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            "precision_no_jit": precision_score(y_test, y_pred, pos_label=0, zero_division=0),
            "recall_jit": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            "recall_no_jit": recall_score(y_test, y_pred, pos_label=0, zero_division=0),
            "f1_no_jit": f1_score(y_test, y_pred, pos_label=0, zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_prob),
        })

results_df = pd.DataFrame(results)
display(results_df.round(4))


fold,model,train_calls,test_calls,train_vessels,test_vessels,accuracy,balanced_accuracy,precision_jit,precision_no_jit,recall_jit,recall_no_jit,f1_no_jit,f1_macro,roc_auc
1,Logistic Regression,154,15,99,11,0.9333,0.9,1.0,0.9091,0.8,1.0,0.9524,0.9206,0.86
1,Random Forest,154,15,99,11,0.8667,0.85,0.8,0.9,0.8,0.9,0.9,0.85,0.86
1,XGBoost,154,15,99,11,0.8,0.8,0.6667,0.8889,0.8,0.8,0.8421,0.7847,0.82
2,Logistic Regression,156,13,101,9,0.8462,0.8,1.0,0.8,0.6,1.0,0.8889,0.8194,0.875
2,Random Forest,156,13,101,9,0.8462,0.8,1.0,0.8,0.6,1.0,0.8889,0.8194,0.825
2,XGBoost,156,13,101,9,0.6923,0.675,0.6,0.75,0.6,0.75,0.75,0.675,0.75
3,Logistic Regression,154,15,101,9,0.8,0.7946,0.8333,0.7778,0.7143,0.875,0.8235,0.7964,0.75
3,Random Forest,154,15,101,9,0.7333,0.7321,0.7143,0.75,0.7143,0.75,0.75,0.7321,0.6964
3,XGBoost,154,15,101,9,0.8,0.7946,0.8333,0.7778,0.7143,0.875,0.8235,0.7964,0.7679
4,Logistic Regression,149,20,96,14,0.85,0.8542,0.7778,0.9091,0.875,0.8333,0.8696,0.8465,0.7813


## 7. Comparación de los tres modelos

Se comparan las medias de los 5 folds, no el mejor fold aislado.

Prioridad:
1. `recall_no_jit` — detectar escalas que realmente serán NO_JIT;
2. `balanced_accuracy` — equilibrio entre ambas clases;
3. `f1_macro`;
4. `roc_auc`.

También se muestran precisión y F1 de NO_JIT para comprobar que el modelo no consigue alto recall simplemente prediciendo demasiados NO_JIT.


In [0]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_jit",
    "precision_no_jit",
    "recall_jit",
    "recall_no_jit",
    "f1_no_jit",
    "f1_macro",
    "roc_auc",
]

summary = results_df.groupby("model")[metric_columns].agg(["mean", "std"]).round(4)
display(summary)

ranking = (
    results_df
    .groupby("model")[metric_columns]
    .mean()
    .reset_index()
    .sort_values(
        ["recall_no_jit", "balanced_accuracy", "f1_macro", "roc_auc"],
        ascending=False
    )
)

display(ranking.round(4))

best_model_name = ranking.iloc[0]["model"]
print("Modelo ganador según prioridad NO_JIT:", best_model_name)


"('accuracy', 'mean')","('accuracy', 'std')","('balanced_accuracy', 'mean')","('balanced_accuracy', 'std')","('precision_jit', 'mean')","('precision_jit', 'std')","('precision_no_jit', 'mean')","('precision_no_jit', 'std')","('recall_jit', 'mean')","('recall_jit', 'std')","('recall_no_jit', 'mean')","('recall_no_jit', 'std')","('f1_no_jit', 'mean')","('f1_no_jit', 'std')","('f1_macro', 'mean')","('f1_macro', 'std')","('roc_auc', 'mean')","('roc_auc', 'std')"
0.7339,0.1548,0.7311,0.1462,0.8013,0.1667,0.692,0.169,0.6542,0.1738,0.8081,0.1795,0.7406,0.163,0.7256,0.1533,0.7146,0.1401
0.7144,0.1381,0.7116,0.1287,0.7583,0.1361,0.6848,0.1653,0.6542,0.1738,0.769,0.1523,0.7192,0.1475,0.7054,0.1353,0.7111,0.1535
0.6982,0.1254,0.6999,0.129,0.7101,0.1112,0.6871,0.1864,0.6759,0.1829,0.7238,0.1503,0.6976,0.1494,0.6894,0.1283,0.7152,0.1333


model,accuracy,balanced_accuracy,precision_jit,precision_no_jit,recall_jit,recall_no_jit,f1_no_jit,f1_macro,roc_auc
Logistic Regression,0.7339,0.7311,0.8013,0.692,0.6542,0.8081,0.7406,0.7256,0.7146
Random Forest,0.7144,0.7116,0.7583,0.6848,0.6542,0.769,0.7192,0.7054,0.7111
XGBoost,0.6982,0.6999,0.7101,0.6871,0.6759,0.7238,0.6976,0.6894,0.7152


Modelo ganador según prioridad NO_JIT: Logistic Regression


In [0]:
for model_name, cm in confusions.items():
    print("\n", model_name)
    cm_df = pd.DataFrame(cm,columns=["PRED_NO_JIT", "PRED_JIT"])
    cm_df.insert(0,"REAL",["REAL_NO_JIT", "REAL_JIT"])
    display(cm_df)


 Logistic Regression


REAL,PRED_NO_JIT,PRED_JIT
REAL_NO_JIT,68,16
REAL_JIT,30,55



 Random Forest


REAL,PRED_NO_JIT,PRED_JIT
REAL_NO_JIT,65,19
REAL_JIT,30,55



 XGBoost


REAL,PRED_NO_JIT,PRED_JIT
REAL_NO_JIT,61,23
REAL_JIT,28,57


## 8. Segunda fase: reducir variables del modelo ganador

La validación anterior elige el **algoritmo**. Ahora comprobamos si ese mismo modelo puede funcionar igual o mejor con menos variables.

Para clasificación se utiliza **Permutation Importance**.

Para evitar usar el test para seleccionar variables:
1. dentro del train de cada fold se crea un pequeño `inner validation`;
2. el ganador se entrena en `inner train`;
3. se permuta cada variable y se mide cuánto empeora `recall_NO_JIT`;
4. se conservan variables con importancia media `>= 0.01`;
5. el modelo reducido se reentrena con todo el train exterior;
6. se evalúa en el test exterior, que no participó en la selección.

`destination_port_code` se conserva siempre porque el modelo actual es global para los tres puertos.


In [0]:
NO_JIT_SCORER = make_scorer(
    recall_score,
    pos_label=0,
    zero_division=0
)

FORCED_FEATURES = [
    c for c in ["destination_port_code"]
    if c in final_features
]

def valid_inner_group_split(X_train, y_train, groups_train):
    # Busca un split interno por MMSI con ambas clases en train y validación.
    for seed in range(BASE_SEED, BASE_SEED + 100):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=INNER_VALIDATION_SIZE,
            random_state=seed
        )
        inner_train_idx, inner_val_idx = next(
            splitter.split(X_train, y_train, groups=groups_train)
        )

        if (
            y_train.iloc[inner_train_idx].nunique() == 2
            and y_train.iloc[inner_val_idx].nunique() == 2
        ):
            return inner_train_idx, inner_val_idx

    raise RuntimeError("No se encontró un split interno válido con ambas clases.")

reduced_results = []
selection_rows = []

cv_reduction = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=BASE_SEED
)

for fold, (train_idx, test_idx) in enumerate(
    cv_reduction.split(X, y, groups),
    start=1
):
    X_outer_train = X.iloc[train_idx].copy()
    X_outer_test = X.iloc[test_idx].copy()
    y_outer_train = y.iloc[train_idx].copy()
    y_outer_test = y.iloc[test_idx].copy()
    groups_outer_train = groups.iloc[train_idx].copy()

    inner_train_idx, inner_val_idx = valid_inner_group_split(
        X_outer_train,
        y_outer_train,
        groups_outer_train
    )

    X_inner_train = X_outer_train.iloc[inner_train_idx]
    X_inner_val = X_outer_train.iloc[inner_val_idx]
    y_inner_train = y_outer_train.iloc[inner_train_idx]
    y_inner_val = y_outer_train.iloc[inner_val_idx]

    # 1) Modelo ganador con todas las features, solo dentro del train exterior.
    importance_model = build_model(best_model_name, final_features)
    importance_model.fit(X_inner_train, y_inner_train)

    # 2) Importancia: caída de recall_NO_JIT al permutar cada variable.
    perm = permutation_importance(
        importance_model,
        X_inner_val,
        y_inner_val,
        scoring=NO_JIT_SCORER,
        n_repeats=N_PERMUTATION_REPEATS,
        random_state=BASE_SEED + fold,
        n_jobs=1
    )

    fold_importance = pd.DataFrame({
        "feature": final_features,
        "importance": perm.importances_mean
    }).sort_values("importance", ascending=False)

    reduced_features = fold_importance.loc[
        fold_importance["importance"] >= MIN_PERMUTATION_IMPORTANCE,
        "feature"
    ].tolist()

    # Puerto se mantiene porque es un modelo global.
    reduced_features = list(dict.fromkeys(
        reduced_features + FORCED_FEATURES
    ))

    # Evita dejar un modelo excesivamente pequeño por ruido de una sola partición.
    if len(reduced_features) < min(MIN_SELECTED_FEATURES, len(final_features)):
        for feature in fold_importance["feature"]:
            if feature not in reduced_features:
                reduced_features.append(feature)
            if len(reduced_features) >= min(MIN_SELECTED_FEATURES, len(final_features)):
                break

    for row in fold_importance.itertuples(index=False):
        selection_rows.append({
            "fold": fold,
            "feature": row.feature,
            "importance_recall_no_jit": row.importance,
            "selected": row.feature in reduced_features
        })

    # 3) Reentrenar versión reducida con TODO el train exterior.
    reduced_model = build_model(best_model_name, reduced_features)
    reduced_model.fit(
        X_outer_train[reduced_features],
        y_outer_train
    )

    y_pred = reduced_model.predict(X_outer_test[reduced_features])
    y_prob = reduced_model.predict_proba(X_outer_test[reduced_features])[:, 1]

    reduced_results.append({
        "fold": fold,
        "features": len(reduced_features),
        "selected_features": ", ".join(reduced_features),
        "accuracy": accuracy_score(y_outer_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_outer_test, y_pred),
        "precision_jit": precision_score(y_outer_test, y_pred, pos_label=1, zero_division=0),
        "precision_no_jit": precision_score(y_outer_test, y_pred, pos_label=0, zero_division=0),
        "recall_jit": recall_score(y_outer_test, y_pred, pos_label=1, zero_division=0),
        "recall_no_jit": recall_score(y_outer_test, y_pred, pos_label=0, zero_division=0),
        "f1_no_jit": f1_score(y_outer_test, y_pred, pos_label=0, zero_division=0),
        "f1_macro": f1_score(y_outer_test, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_outer_test, y_prob),
    })

reduced_results_df = pd.DataFrame(reduced_results)
selection_df = pd.DataFrame(selection_rows)

display(reduced_results_df.round(4))


fold,features,selected_features,accuracy,balanced_accuracy,precision_jit,precision_no_jit,recall_jit,recall_no_jit,f1_no_jit,f1_macro,roc_auc
1,4,"speed_over_ground_knots, navigation_status_code, vessel_length_meters, destination_port_code",0.8667,0.85,0.8,0.9,0.8,0.9,0.9,0.85,0.84
2,4,"speed_over_ground_knots, course_over_ground_degrees, navigation_status_code, destination_port_code",0.8462,0.8,1.0,0.8,0.6,1.0,0.8889,0.8194,0.9
3,5,"navigation_status_code, speed_over_ground_knots, distance_to_service_nm, course_over_ground_degrees, destination_port_code",0.8,0.7946,0.8333,0.7778,0.7143,0.875,0.8235,0.7964,0.9107
4,4,"speed_over_ground_knots, navigation_status_code, distance_to_service_nm, destination_port_code",0.85,0.8542,0.7778,0.9091,0.875,0.8333,0.8696,0.8465,0.7708
5,3,"speed_over_ground_knots, navigation_status_code, destination_port_code",0.7143,0.7222,0.8,0.6364,0.6667,0.7778,0.7,0.7136,0.7546
6,5,"speed_over_ground_knots, navigation_status_code, vessel_length_meters, vessel_draught_meters, destination_port_code",0.4444,0.45,0.5,0.4,0.4,0.5,0.4444,0.4444,0.5625
7,4,"speed_over_ground_knots, navigation_status_code, distance_to_service_nm, destination_port_code",0.6471,0.6458,0.625,0.6667,0.625,0.6667,0.6667,0.6458,0.7083
8,6,"speed_over_ground_knots, navigation_status_code, course_over_ground_degrees, vessel_length_meters, vessel_draught_meters, destination_port_code",0.5625,0.5952,0.75,0.5,0.3333,0.8571,0.6316,0.5466,0.4603
9,3,"speed_over_ground_knots, navigation_status_code, destination_port_code",0.875,0.9,1.0,0.75,0.8,1.0,0.8571,0.873,0.8
10,3,"speed_over_ground_knots, navigation_status_code, destination_port_code",0.6667,0.6494,0.7273,0.5714,0.7273,0.5714,0.5714,0.6494,0.6753


## 9. Comparación: ganador completo vs ganador reducido

La tabla compara el modelo ganador original con el procedimiento de reducción validado en los mismos 10 folds exteriores.

Si la versión reducida mantiene o mejora `recall_no_jit` y no empeora claramente `balanced_accuracy`, se prefiere por ser más simple.


In [0]:
full_best = (
    results_df[results_df["model"] == best_model_name]
    [metric_columns]
    .mean()
)

reduced_best = reduced_results_df[metric_columns].mean()

comparison_df = pd.DataFrame([
    {"version": "FULL", **full_best.to_dict()},
    {"version": "REDUCED", **reduced_best.to_dict()},
])

display(comparison_df.round(4))

print(
    "Δ recall_NO_JIT:",
    round(reduced_best["recall_no_jit"] - full_best["recall_no_jit"], 4)
)
print(
    "Δ balanced_accuracy:",
    round(reduced_best["balanced_accuracy"] - full_best["balanced_accuracy"], 4)
)


version,accuracy,balanced_accuracy,precision_jit,precision_no_jit,recall_jit,recall_no_jit,f1_no_jit,f1_macro,roc_auc
FULL,0.7339,0.7311,0.8013,0.692,0.6542,0.8081,0.7406,0.7256,0.7146
REDUCED,0.7273,0.7261,0.7813,0.6911,0.6542,0.7981,0.7353,0.7185,0.7383


Δ recall_NO_JIT: -0.01
Δ balanced_accuracy: -0.005


## 10. Variables estables y entrenamiento final

Una variable se considera estable si fue seleccionada en al menos **6 de los 10 folds**. `destination_port_code` se mantiene siempre.

El modelo final se ajusta con todas las escalas disponibles porque, una vez terminada la validación, ya no necesitamos reservar esos casos para estimar métricas. Las métricas válidas son las obtenidas en los folds anteriores, no las del ajuste final.


In [0]:
feature_stability = (
    selection_df
    .groupby("feature")
    .agg(
        selected_folds=("selected", "sum"),
        mean_importance_recall_no_jit=("importance_recall_no_jit", "mean")
    )
    .reset_index()
    .sort_values(
        ["selected_folds", "mean_importance_recall_no_jit"],
        ascending=False
    )
)

display(feature_stability.round(4))

stable_features = feature_stability.loc[
    feature_stability["selected_folds"] >= MIN_FEATURE_FOLDS,
    "feature"
].tolist()

stable_features = list(dict.fromkeys(
    stable_features + FORCED_FEATURES
))

if len(stable_features) < min(MIN_SELECTED_FEATURES, len(final_features)):
    for feature in feature_stability["feature"]:
        if feature not in stable_features:
            stable_features.append(feature)
        if len(stable_features) >= min(MIN_SELECTED_FEATURES, len(final_features)):
            break

print("Features estables finales:", stable_features)

# Se usa la versión reducida si no pierde recall_NO_JIT y mantiene
# balanced_accuracy con una tolerancia máxima de 0.05.
MAX_RECALL_DROP = 0.05
MAX_BALANCED_ACCURACY_DROP = 0.05

use_reduced = (
    reduced_best["recall_no_jit"] >= full_best["recall_no_jit"] - MAX_RECALL_DROP
    and
    reduced_best["balanced_accuracy"] >= full_best["balanced_accuracy"] - MAX_BALANCED_ACCURACY_DROP
)

training_features = stable_features if use_reduced else final_features
final_version = "REDUCED" if use_reduced else "FULL"

best_model = build_model(best_model_name, training_features)
best_model.fit(pdf[training_features], y)

print("Versión final:", final_version)
print("Modelo:", best_model_name)
print("Features:", training_features)
print(f"[OK] Entrenado con {len(pdf)} port calls y {groups.nunique()} MMSI.")


feature,selected_folds,mean_importance_recall_no_jit
speed_over_ground_knots,10,0.207
navigation_status_code,10,0.1479
destination_port_code,10,0.0022
distance_to_service_nm,3,0.0204
vessel_length_meters,3,0.0104
course_over_ground_degrees,3,0.01
vessel_draught_meters,2,0.0037
vessel_beam_meters,0,-0.0109


Features estables finales: ['speed_over_ground_knots', 'navigation_status_code', 'destination_port_code']
Versión final: REDUCED
Modelo: Logistic Regression
Features: ['speed_over_ground_knots', 'navigation_status_code', 'destination_port_code']
[OK] Entrenado con 169 port calls y 110 MMSI.


### Nota metodológica

El ajuste final con todos los datos **no sirve para calcular nuevas métricas**; solo deja preparado el modelo elegido.  
La estimación de rendimiento procede de los datos que quedaron fuera del entrenamiento en cada fold.

Cuando exista más histórico, la comprobación más fuerte será evaluar este modelo sobre un periodo temporal posterior completamente nuevo.


## 11. Persistir modelo final en MLflow

La validación ya ha terminado. Esta celda **no vuelve a comparar modelos**: guarda el `best_model` final para reutilizarlo en inferencia sin reentrenar.

Se registran también la versión elegida (`FULL`/`REDUCED`), las features utilizadas y las métricas medias obtenidas en validación cruzada.


In [0]:
import mlflow, mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

REGISTERED_MODEL_NAME = registered_model_name
X_final = pdf[training_features]
final_cv_metrics = reduced_best if final_version == "REDUCED" else full_best
signature = infer_signature(X_final, best_model.predict(X_final))

with mlflow.start_run(run_name="vessel_jit_classifier_training") as run:
    mlflow.log_params({"model_name": best_model_name, "model_version": final_version, "target": TARGET, "n_splits": N_SPLITS, "training_port_calls": len(pdf), "training_vessels": groups.nunique(), "feature_count": len(training_features)})
    mlflow.log_dict({"features": training_features}, "features.json")
    for metric in metric_columns: mlflow.log_metric(f"cv_{metric}", float(final_cv_metrics[metric]))
    mlflow.sklearn.log_model(best_model, "model", signature=signature, input_example=X_final.head(5))
    RUN_MODEL_URI = f"runs:/{run.info.run_id}/model"

client = MlflowClient()
version = mlflow.register_model(RUN_MODEL_URI, REGISTERED_MODEL_NAME)

candidate_recall = float(final_cv_metrics["recall_no_jit"])
candidate_balanced_accuracy = float(final_cv_metrics["balanced_accuracy"])

try:
    champion = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, registered_model_alias)
    champion_metrics = client.get_run(champion.run_id).data.metrics
    champion_recall = champion_metrics["cv_recall_no_jit"]
    champion_balanced_accuracy = champion_metrics["cv_balanced_accuracy"]

    promote = candidate_recall > champion_recall or (candidate_recall == champion_recall and candidate_balanced_accuracy > champion_balanced_accuracy)

    if promote:
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, registered_model_alias, version.version)
        print(f"[OK] Nueva version {version.version} promovida a {registered_model_alias}.")
    else:
        print(f"[OK] Version {version.version} registrada pero no promovida. Se mantiene Champion v{champion.version}.")

except Exception:
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, registered_model_alias, version.version)
    print(f"[OK] No existia Champion previo. Version {version.version} promovida a {registered_model_alias}.")


MODEL_URI = registered_model_uri

print(f"[OK] {REGISTERED_MODEL_NAME} | version={version.version} | alias=Champion")
print("MODEL_URI =", MODEL_URI)


Successfully registered model 'masterxyz002dbr.gold.vessel_jit_classifier'.
Created version '1' of model 'masterxyz002dbr.gold.vessel_jit_classifier'.


[OK] masterxyz002dbr.gold.vessel_jit_classifier | version=1 | alias=Champion
MODEL_URI = models:/masterxyz002dbr.gold.vessel_jit_classifier@Champion
